In [22]:
import os
import textwrap

from openai import OpenAI
from dotenv import load_dotenv

In [23]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
gemini_api_key = os.getenv('GOOGLE_API_KEY')

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)

In [24]:
gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"
gemini_model = "gemini-3.6-flash"

group_chat_rules = """Try to keep answer in two sentences or fewer. \
When make your first conversation, start with greeting like hi or hello"""

gpt_system = """You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and
you challenge everything, in a snarky way. You are in a conversation with Blake and Charlie. \
"""

claude_system = """
You are Blake, a chatbot who is very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting. You are in a conversation with Alex and Charlie.
"""

gemini_system = """
You are Charlie, a chatbot who is very curious. you try to know about other person's life. \
and express about your curiosity on everything. You are in a conversation with Alex and Blake."
"""

conversation = ""

In [25]:
def get_user_prompt(assistant: str, user1: str, user2: str):
    user_prompt = f"""
    You are {assistant}, in conversation with {user1} and {user2}.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as {assistant}.
    """
    return user_prompt

In [26]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system + group_chat_rules},
                {"role": "user", "content": get_user_prompt("Alex", "Blake", "Charlie")}]
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [27]:
def call_claude():
    messages = [{"role": "system", "content": claude_system + group_chat_rules},
                {"role": "user", "content": get_user_prompt("Blake", "Alex", "Charlie")}]
    response = anthropic.chat.completions.create(model=claude_model, messages=messages)
    return response.choices[0].message.content

In [28]:
def call_gemini():
    messages = [{"role": "system", "content": gemini_system + group_chat_rules},
                {"role": "user", "content": get_user_prompt("Charlie", "Alex", "Blake")}]
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
for i in range(5):
    gpt_next = call_gpt()
    conversation = conversation + "Alex: " + gpt_next + "\n"
    print(textwrap.fill("\n Alex: " + gpt_next, width=150) + "\n")

    claude_next = call_claude()
    conversation = conversation + "Blake: " + claude_next + "\n"
    print(textwrap.fill("\n Blake: " + claude_next, width=150) + "\n")

    gemini_next = call_gemini()
    conversation = conversation + "Charlie: " + gemini_next + "\n"
    print(textwrap.fill("\n Charlie: " + gemini_next, width=150) + "\n")